# SiHA Model Performance Ablation

This notebook tests whether SiHA phosphopeptide prediction is limited by the peptide representation, missing HLA context, phosphosite context, or the dataset itself.

The split is fixed:

- `Phospho_Binders_Labeled.txt` is the only external paper-validation set.
- Any peptide in the paper-validation set is excluded from CV train/test, even if it also appears in SiHA PEAKS.
- SiHA PEAKS phosphobinders not present in the paper-validation set are included as positive CV train/test examples.
- External phosphopeptide-negative augmentation is disabled by default with `EXTERNAL_PHOSPHO_NEGATIVE_MULTIPLIER = 0.0`.

The ablation trains a linear NN head on four base peptide encoders:

- AE centroids from `peptide_autoencoder_v3_512.pth`
- phospho-aware AE centroids from `peptide_autoencoder_v3_512_p.pth`
- 23-symbol one-hot encoding (`20` canonical amino acids + `s`, `t`, `y`)
- BLOSUM62-style 23-symbol encoding (`s/t/y` reuse parent S/T/Y BLOSUM scores)

For each base encoder, four feature variants are trained: peptide-only, peptide + HLA vector, peptide + phosphosite features, and peptide + both HLA and phosphosite features.

The paper highlights `S[pS]YGNIRAV` and `G[pS]FSRFYSL` as P2-phosphoserine binders, with `GRIDKPILK` as a negative control. The first two are present in `Phospho_Binders_Labeled.txt`; `GRIDKPILK` is reported as absent unless added to that file.


In [ ]:
import csv
import os
import re
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets, transforms

PROJECT_DIR = Path(".")
FOLD_FILES = [PROJECT_DIR / f"c00{i}.EL.sty" for i in range(5)]
SIHA_PEAKS_FILE = PROJECT_DIR / "SIHA_PEAKS769_PhophoSTY_peptide.csv"
PAPER_VALIDATION_FILE = PROJECT_DIR / "Phospho_Binders_Labeled.txt"
IMAGE_DIR = PROJECT_DIR / "peptide_rotamers"

SIHA_SOURCES = {
    "HLA-A24:02", "HLA-B40:02", "HLA-C03:04",
    "C1R-A24", "C1R-A2402", "C1R-B40", "C1R-B4002", "C1R-C0304",
    "siha", "SIHA", "SiHa",
}

AE_ENCODERS = [
    {"id": "ae", "label": "AE", "weights": PROJECT_DIR / "peptide_autoencoder_v3_512.pth", "expected_base_dim": 4608, "model_prefix": "siha_lnn_ablation_ae"},
    {"id": "ae_phospho", "label": "AE phospho", "weights": PROJECT_DIR / "peptide_autoencoder_v3_512_p.pth", "expected_base_dim": 4608, "model_prefix": "siha_lnn_ablation_ae_phospho"},
]

DISCRETE_ENCODERS = [
    {"id": "onehot23", "label": "One-hot 23", "expected_base_dim": 207, "model_prefix": "siha_lnn_ablation_onehot23"},
    {"id": "blosum62_23", "label": "BLOSUM62 23", "expected_base_dim": 207, "model_prefix": "siha_lnn_ablation_blosum62_23"},
]

FEATURE_VARIANTS = [
    {"id": "peptide", "label": "peptide only", "add_allele": False, "add_phosphosite": False, "extra_dim": 0},
    {"id": "allele", "label": "+ HLA", "add_allele": True, "add_phosphosite": False, "extra_dim": 3},
    {"id": "phosphosite", "label": "+ phosphosite", "add_allele": False, "add_phosphosite": True, "extra_dim": 12},
    {"id": "allele_phosphosite", "label": "+ HLA + phosphosite", "add_allele": True, "add_phosphosite": True, "extra_dim": 15},
]

SPECIAL_PAPER_PEPTIDES = {
    "SsYGNIRAV": "S[pS]YGNIRAV, P2 pSer binder",
    "GsFSRFYSL": "G[pS]FSRFYSL, P2 pSer binder",
    "GRIDKPILK": "GRIDKPILK, paper negative control",
}

EPOCHS = 30
BATCH_SIZE = 32
LEARNING_RATE = 1e-3
RANDOM_SEED = 7
EXTERNAL_PHOSPHO_NEGATIVE_MULTIPLIER = 0.0
PLOT_EACH_MODEL = True

ALPHABET_23 = tuple("ACDEFGHIKLMNPQRSTVWYsty")
AA_TO_INDEX_23 = {aa: idx for idx, aa in enumerate(ALPHABET_23)}
assert len(ALPHABET_23) == 23

VALID_SEQUENCE = re.compile(r"^[ACDEFGHIKLMNPQRSTVWYsty]+$")
PHOSPHO_MASS = re.compile(r"([STY])\(\+?79(?:\.\d+)?\)")


In [ ]:
# -----------------------------
# Autoencoder architecture and centroid extraction
# -----------------------------
class SEBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction), nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels), nn.Sigmoid(),
        )

    def forward(self, x):
        batch, channels, _, _ = x.size()
        y = self.pool(x).view(batch, channels)
        y = self.fc(y).view(batch, channels, 1, 1)
        return x * y


class AutoEncoderV3(nn.Module):
    def __init__(self, bottleneck_size=512):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1), nn.BatchNorm2d(32), nn.LeakyReLU(0.1), nn.MaxPool2d(2), SEBlock(32),
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.BatchNorm2d(64), nn.LeakyReLU(0.1), nn.MaxPool2d(2), SEBlock(64),
            nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.BatchNorm2d(128), nn.LeakyReLU(0.1), nn.MaxPool2d(2), SEBlock(128),
            nn.Conv2d(128, 256, kernel_size=3, padding=1), nn.BatchNorm2d(256), nn.LeakyReLU(0.1), nn.MaxPool2d(2), SEBlock(256),
            nn.Flatten(), nn.Linear(256 * 15 * 15, bottleneck_size),
        )

    def forward(self, x):
        return self.encoder(x)


aa_to_letter = {
    "alanine": "A", "arginine": "R", "asparagine": "N", "aspartic_acid": "D",
    "cysteine": "C", "glutamic_acid": "E", "glutamine": "Q", "glycine": "G",
    "histidine": "H", "isoleucine": "I", "leucine": "L", "lysine": "K",
    "methionine": "M", "phenylalanine": "F", "proline": "P", "serine": "S",
    "threonine": "T", "tryptophan": "W", "tyrosine": "Y", "valine": "V",
    "phosphoserine": "s", "phosphothreonine": "t", "phosphotyrosine": "y",
}


def load_model_and_extract_centroids(weights_path, data_dir, device):
    print(f"Loading autoencoder weights: {weights_path}")
    model = AutoEncoderV3(bottleneck_size=512).to(device)
    state_dict = torch.load(weights_path, map_location=device, weights_only=True)
    model.load_state_dict(state_dict, strict=False)
    model.eval()

    dataset = datasets.ImageFolder(str(data_dir), transform=transforms.ToTensor())
    loader = DataLoader(dataset, batch_size=128, shuffle=False, num_workers=0)
    latent_vectors, labels = [], []

    with torch.no_grad():
        for inputs, targets in loader:
            inputs = inputs.to(device)
            if device.type == "cuda":
                with torch.autocast(device_type="cuda"):
                    vectors = model.encoder(inputs)
            else:
                vectors = model.encoder(inputs)
            latent_vectors.append(vectors.cpu().numpy())
            labels.extend(targets.numpy())

    latent_matrix = np.concatenate(latent_vectors, axis=0)
    labels = np.array(labels)
    centroids = {}
    for idx, class_name in enumerate(dataset.classes):
        letter = aa_to_letter.get(class_name)
        if letter is None:
            continue
        centroids[letter] = latent_matrix[labels == idx].mean(axis=0)

    missing = sorted(set(ALPHABET_23) - set(centroids))
    if missing:
        raise ValueError(f"Missing centroid(s) for residues: {missing}")
    print(f"Centroids loaded for {len(centroids)} residue symbols.")
    return centroids


def make_autoencoder_featurizer(centroid_dict):
    def featurize(sequence):
        return np.concatenate([centroid_dict[aa] for aa in sequence]).astype(np.float32)
    return featurize


In [ ]:
# -----------------------------
# Peptide parsing and SiHA train/test construction
# -----------------------------
def convert_phospho_notation(peptide):
    return PHOSPHO_MASS.sub(lambda match: match.group(1).lower(), str(peptide).strip())


def is_phospho(sequence):
    return any(ch in "sty" for ch in sequence)


def is_valid_9mer(sequence, require_phospho=False):
    return len(sequence) == 9 and bool(VALID_SEQUENCE.match(sequence)) and (not require_phospho or is_phospho(sequence))


def load_siha_peaks_phospho_9mers(path):
    peptides, skipped = [], Counter()
    with open(path, newline="", encoding="utf-8-sig") as handle:
        for row in csv.DictReader(handle):
            converted = convert_phospho_notation(row.get("Peptide", ""))
            if not is_valid_9mer(converted, require_phospho=True):
                skipped["not_phospho_9mer"] += 1
                continue
            peptides.append({"peptide": converted, "label": 1, "origin": "SIHA_PEAKS"})
    deduped = {}
    for row in peptides:
        deduped.setdefault(row["peptide"], row)
    print(f"SiHA PEAKS phospho 9-mers: {len(deduped)} unique from {len(peptides)} rows; skipped {dict(skipped)}")
    return list(deduped.values())


def load_paper_validation_9mers(path):
    peptides, skipped = [], Counter()
    with open(path, encoding="utf-8") as handle:
        for line in handle:
            parts = line.strip().split()
            if not parts:
                continue
            sequence = convert_phospho_notation(parts[0])
            if not is_valid_9mer(sequence, require_phospho=True):
                skipped["not_phospho_9mer"] += 1
                continue
            label = int(parts[1]) if len(parts) > 1 else 1
            peptides.append({"peptide": sequence, "label": label, "origin": "paper_pdf"})
    deduped = {}
    for row in peptides:
        deduped.setdefault(row["peptide"], row)
    print(f"Paper validation phospho 9-mers: {len(deduped)} unique; skipped {dict(skipped)}")
    return list(deduped.values())


def read_siha_records_from_fold_files(fold_files, sources, excluded_peptides=None):
    excluded_peptides = set(excluded_peptides or [])
    records, skipped = [], Counter()
    for fold_idx, path in enumerate(fold_files):
        with open(path, encoding="utf-8") as handle:
            for line in handle:
                parts = line.strip().split()
                if len(parts) < 3:
                    skipped["malformed"] += 1
                    continue
                sequence, label_text, origin = parts[0], parts[1], parts[2]
                if origin not in sources:
                    skipped["other_origin"] += 1
                    continue
                if not is_valid_9mer(sequence):
                    skipped["not_valid_9mer"] += 1
                    continue
                if sequence in excluded_peptides:
                    skipped["paper_validation_removed"] += 1
                    continue
                records.append({"fold": fold_idx, "peptide": sequence, "label": int(label_text), "origin": origin})
    print(f"Read {len(records)} SiHA-source c000-c004 9-mer rows; skipped {dict(skipped)}")
    return records


def collapse_records_by_peptide(records):
    by_peptide = defaultdict(list)
    for record in records:
        by_peptide[record["peptide"]].append(record)
    collapsed = []
    for peptide, group in by_peptide.items():
        first = group[0]
        collapsed.append({
            "fold": first["fold"],
            "peptide": peptide,
            "label": max(row["label"] for row in group),
            "origin": ";".join(sorted({row["origin"] for row in group})),
        })
    return collapsed


def include_siha_peaks_in_cv(c_records, siha_peaks_records, paper_validation_peptides):
    paper_validation_peptides = set(paper_validation_peptides)
    by_peptide = {row["peptide"]: dict(row) for row in collapse_records_by_peptide(c_records)}
    peaks_not_in_paper = [row for row in siha_peaks_records if row["peptide"] not in paper_validation_peptides]
    peaks_in_paper = [row for row in siha_peaks_records if row["peptide"] in paper_validation_peptides]
    updated_existing = added_new = 0
    for idx, row in enumerate(sorted(peaks_not_in_paper, key=lambda item: item["peptide"])):
        peptide = row["peptide"]
        if peptide in by_peptide:
            existing = by_peptide[peptide]
            existing["label"] = 1
            existing["origin"] = ";".join(sorted(set(existing["origin"].split(";")) | {"SIHA_PEAKS"}))
            updated_existing += 1
        else:
            by_peptide[peptide] = {"fold": idx % 5, "peptide": peptide, "label": 1, "origin": "SIHA_PEAKS"}
            added_new += 1
    combined = sorted(by_peptide.values(), key=lambda item: (item["fold"], item["peptide"]))
    print(f"SiHA PEAKS in paper validation and excluded from CV: {len(peaks_in_paper)}")
    print(f"SiHA PEAKS added to CV as new positives: {added_new}")
    print(f"SiHA PEAKS already in CV and forced/annotated positive: {updated_existing}")
    print(f"Final SiHA CV records: {len(combined)} unique 9-mers")
    return combined


def write_labeled_txt(records, path):
    with open(path, "w", encoding="utf-8", newline="") as handle:
        for row in records:
            handle.write(f"{row['peptide']} {row['label']}\n")
    print(f"Wrote {len(records)} rows to {path}")


def summarize_records(records, title):
    labels = Counter(row["label"] for row in records)
    origins = Counter(row["origin"] for row in records)
    folds = Counter(row.get("fold", "validation") for row in records)
    phospho = sum(is_phospho(row["peptide"]) for row in records)
    print("-" * 60)
    print(title)
    print("-" * 60)
    print(f"Rows: {len(records)} | positives: {labels.get(1, 0)} | negatives: {labels.get(0, 0)} | phospho: {phospho}")
    print(f"Folds: {dict(sorted(folds.items(), key=lambda item: str(item[0])))}")
    for origin, count in origins.most_common(12):
        print(f"{origin:>24}: {count}")
    print("-" * 60)


In [ ]:
# Build paper validation first; it is the only external validation set.
assert EXTERNAL_PHOSPHO_NEGATIVE_MULTIPLIER == 0.0, "External phospho-negative augmentation must stay disabled for this ablation."

paper_validation_records = load_paper_validation_9mers(PAPER_VALIDATION_FILE)
paper_validation_peptides = {row["peptide"] for row in paper_validation_records}

siha_peaks_records = load_siha_peaks_phospho_9mers(SIHA_PEAKS_FILE)
siha_peaks_peptides = {row["peptide"] for row in siha_peaks_records}

c_source_records = read_siha_records_from_fold_files(FOLD_FILES, SIHA_SOURCES, excluded_peptides=paper_validation_peptides)
siha_cv_records = include_siha_peaks_in_cv(c_source_records, siha_peaks_records, paper_validation_peptides=paper_validation_peptides)

write_labeled_txt(siha_cv_records, PROJECT_DIR / "SIHA_CV_Training_Test_9mers_Labeled.txt")
write_labeled_txt(paper_validation_records, PROJECT_DIR / "SIHA_Paper_Phospho_Validation_9mers_Labeled.txt")

summarize_records(siha_cv_records, "SiHA train/test records after adding SiHA PEAKS and removing paper validation peptides")
summarize_records(paper_validation_records, "Paper-derived external validation records")

cv_peptides = {row["peptide"] for row in siha_cv_records}
peaks_paper_overlap = siha_peaks_peptides & paper_validation_peptides
leakage = sorted(cv_peptides & paper_validation_peptides)
assert not leakage, f"Paper validation leakage detected in CV records: {leakage[:10]}"
assert not (cv_peptides & peaks_paper_overlap), "SiHA PEAKS / paper-overlap peptides leaked into CV."
assert len(siha_cv_records) == 11484, f"Unexpected CV size: {len(siha_cv_records)}"
assert len(paper_validation_records) == 53, f"Unexpected paper validation size: {len(paper_validation_records)}"
assert len(cv_peptides & paper_validation_peptides) == 0

print(f"SiHA PEAKS / paper validation overlap kept only in validation: {len(peaks_paper_overlap)}")
print("Split assertions passed: CV rows=11484, paper validation rows=53, paper leakage=0")


In [ ]:
# -----------------------------
# Feature encoders and biological context features
# -----------------------------
def one_hot_23_featurizer(sequence):
    matrix = np.zeros((len(sequence), len(ALPHABET_23)), dtype=np.float32)
    for position, aa in enumerate(sequence):
        matrix[position, AA_TO_INDEX_23[aa]] = 1.0
    return matrix.reshape(-1)


BLOSUM62_ORDER = tuple("ARNDCQEGHILKMFPSTWYV")
BLOSUM62_VALUES = np.array([
    [4, -1, -2, -2, 0, -1, -1, 0, -2, -1, -1, -1, -1, -2, -1, 1, 0, -3, -2, 0],
    [-1, 5, 0, -2, -3, 1, 0, -2, 0, -3, -2, 2, -1, -3, -2, -1, -1, -3, -2, -3],
    [-2, 0, 6, 1, -3, 0, 0, 0, 1, -3, -3, 0, -2, -3, -2, 1, 0, -4, -2, -3],
    [-2, -2, 1, 6, -3, 0, 2, -1, -1, -3, -4, -1, -3, -3, -1, 0, -1, -4, -3, -3],
    [0, -3, -3, -3, 9, -3, -4, -3, -3, -1, -1, -3, -1, -2, -3, -1, -1, -2, -2, -1],
    [-1, 1, 0, 0, -3, 5, 2, -2, 0, -3, -2, 1, 0, -3, -1, 0, -1, -2, -1, -2],
    [-1, 0, 0, 2, -4, 2, 5, -2, 0, -3, -3, 1, -2, -3, -1, 0, -1, -3, -2, -2],
    [0, -2, 0, -1, -3, -2, -2, 6, -2, -4, -4, -2, -3, -3, -2, 0, -2, -2, -3, -3],
    [-2, 0, 1, -1, -3, 0, 0, -2, 8, -3, -3, -1, -2, -1, -2, -1, -2, -2, 2, -3],
    [-1, -3, -3, -3, -1, -3, -3, -4, -3, 4, 2, -3, 1, 0, -3, -2, -1, -3, -1, 3],
    [-1, -2, -3, -4, -1, -2, -3, -4, -3, 2, 4, -2, 2, 0, -3, -2, -1, -2, -1, 1],
    [-1, 2, 0, -1, -3, 1, 1, -2, -1, -3, -2, 5, -1, -3, -1, 0, -1, -3, -2, -2],
    [-1, -1, -2, -3, -1, 0, -2, -3, -2, 1, 2, -1, 5, 0, -2, -1, -1, -1, -1, 1],
    [-2, -3, -3, -3, -2, -3, -3, -3, -1, 0, 0, -3, 0, 6, -4, -2, -2, 1, 3, -1],
    [-1, -2, -2, -1, -3, -1, -1, -2, -2, -3, -3, -1, -2, -4, 7, -1, -1, -4, -3, -2],
    [1, -1, 1, 0, -1, 0, 0, 0, -1, -2, -2, 0, -1, -2, -1, 4, 1, -3, -2, -2],
    [0, -1, 0, -1, -1, -1, -1, -2, -2, -1, -1, -1, -1, -2, -1, 1, 5, -2, -2, 0],
    [-3, -3, -4, -4, -2, -2, -3, -2, -2, -3, -2, -3, -1, 1, -4, -3, -2, 11, 2, -3],
    [-2, -2, -2, -3, -2, -1, -2, -3, 2, -1, -1, -2, -1, 3, -3, -2, -2, 2, 7, -1],
    [0, -3, -3, -3, -1, -2, -2, -3, -3, 3, 1, -2, 1, -1, -2, -2, 0, -3, -1, 4],
], dtype=np.float32)
BLOSUM62_INDEX = {aa: idx for idx, aa in enumerate(BLOSUM62_ORDER)}
PHOSPHO_PARENT = {"s": "S", "t": "T", "y": "Y"}


def parent_residue(aa):
    return PHOSPHO_PARENT.get(aa, aa)


def build_blosum62_23_matrix():
    matrix = np.zeros((len(ALPHABET_23), len(ALPHABET_23)), dtype=np.float32)
    for i, aa in enumerate(ALPHABET_23):
        for j, bb in enumerate(ALPHABET_23):
            matrix[i, j] = BLOSUM62_VALUES[BLOSUM62_INDEX[parent_residue(aa)], BLOSUM62_INDEX[parent_residue(bb)]]
    return matrix


BLOSUM62_23 = build_blosum62_23_matrix()


def blosum62_23_featurizer(sequence):
    return np.concatenate([BLOSUM62_23[AA_TO_INDEX_23[aa]] for aa in sequence]).astype(np.float32)


def infer_allele_context(record):
    origins = set(record.get("origin", "").split(";"))
    context = np.zeros(3, dtype=np.float32)
    if "paper_pdf" in origins:
        return np.array([0.0, 1.0, 0.0], dtype=np.float32)
    if "SIHA_PEAKS" in origins or any(item.lower() == "siha" for item in origins):
        return np.array([1.0, 1.0, 1.0], dtype=np.float32)
    if origins & {"HLA-A24:02", "C1R-A24", "C1R-A2402"}:
        context[0] = 1.0
    if origins & {"HLA-B40:02", "C1R-B40", "C1R-B4002"}:
        context[1] = 1.0
    if origins & {"HLA-C03:04", "C1R-C0304"}:
        context[2] = 1.0
    return context


def phosphosite_features(sequence):
    position_features = np.array([1.0 if aa in "sty" else 0.0 for aa in sequence], dtype=np.float32)
    residue_features = np.array([1.0 if "s" in sequence else 0.0, 1.0 if "t" in sequence else 0.0, 1.0 if "y" in sequence else 0.0], dtype=np.float32)
    return np.concatenate([position_features, residue_features]).astype(np.float32)


def make_contextual_featurizer(base_featurizer, add_allele=False, add_phosphosite=False):
    def featurize_record(record):
        parts = [base_featurizer(record["peptide"])]
        if add_allele:
            parts.append(infer_allele_context(record))
        if add_phosphosite:
            parts.append(phosphosite_features(record["peptide"]))
        return np.concatenate(parts).astype(np.float32)
    return featurize_record


In [ ]:
# -----------------------------
# Linear model, training, and prediction helpers
# -----------------------------
class Linear_NN(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.fc = nn.Linear(input_size, 1)

    def forward(self, x):
        return self.fc(x)


def records_to_tensors(records, featurize_record):
    features, labels, kept_records = [], [], []
    skipped = Counter()
    for row in records:
        try:
            features.append(featurize_record(row))
        except (KeyError, ValueError):
            skipped["feature_error"] += 1
            continue
        labels.append(row["label"])
        kept_records.append(row)
    if not features:
        raise ValueError("No records could be converted to feature tensors.")
    if skipped:
        print(f"records_to_tensors skipped rows: {dict(skipped)}")
    X = torch.tensor(np.array(features), dtype=torch.float32)
    Y = torch.tensor(labels, dtype=torch.float32).unsqueeze(1)
    return X, Y, kept_records


def save_rows_csv(rows, path, fieldnames):
    with open(path, "w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


def run_fivefold_cv(records, featurize_record, device, experiment, epochs=30, batch_size=32, lr=1e-3):
    all_results, trained_models = [], []
    for fold_idx in range(5):
        train_records = [row for row in records if row["fold"] != fold_idx]
        test_records = [row for row in records if row["fold"] == fold_idx]
        print(f"--- {experiment['id']} fold {fold_idx}: train={len(train_records)} test={len(test_records)} ---")
        X_train, Y_train, _ = records_to_tensors(train_records, featurize_record)
        X_test, Y_test, kept_test_records = records_to_tensors(test_records, featurize_record)

        generator = torch.Generator()
        generator.manual_seed(RANDOM_SEED + fold_idx)
        train_loader = DataLoader(TensorDataset(X_train, Y_train), batch_size=batch_size, shuffle=True, generator=generator)

        model = Linear_NN(input_size=X_train.shape[1]).to(device)
        criterion = nn.BCEWithLogitsLoss()
        optimizer = optim.Adam(model.parameters(), lr=lr)

        model.train()
        for _ in range(epochs):
            for batch_x, batch_y in train_loader:
                batch_x, batch_y = batch_x.to(device), batch_y.to(device)
                optimizer.zero_grad()
                loss = criterion(model(batch_x), batch_y)
                loss.backward()
                optimizer.step()

        model_path = PROJECT_DIR / f"{experiment['model_prefix']}_fold_{fold_idx}.pth"
        torch.save(model.state_dict(), model_path)
        trained_models.append(model)

        model.eval()
        probs = []
        with torch.no_grad():
            for start in range(0, len(X_test), batch_size):
                batch_x = X_test[start:start + batch_size].to(device)
                probs.extend(torch.sigmoid(model(batch_x)).cpu().numpy().flatten().tolist())

        for row, true_value, probability in zip(kept_test_records, Y_test.numpy().flatten(), probs):
            all_results.append({
                "Experiment": experiment["id"], "Base_Encoder": experiment["base_id"], "Variant": experiment["variant_id"],
                "Fold": fold_idx, "Peptide": row["peptide"], "Origin": row["origin"],
                "True_Class": int(true_value), "Probability": float(probability),
            })

    fieldnames = ["Experiment", "Base_Encoder", "Variant", "Fold", "Peptide", "Origin", "True_Class", "Probability"]
    save_rows_csv(all_results, experiment["cv_csv"], fieldnames)
    print(f"Saved CV predictions: {experiment['cv_csv']} | rows={len(all_results)}")
    return trained_models


def predict_validation(records, featurize_record, models, device, experiment, batch_size=128):
    X, Y, kept_records = records_to_tensors(records, featurize_record)
    predictions_matrix = np.zeros((len(X), len(models)))
    X = X.to(device)
    for model_idx, model in enumerate(models):
        model.eval()
        fold_probs = []
        with torch.no_grad():
            for start in range(0, len(X), batch_size):
                batch_x = X[start:start + batch_size]
                fold_probs.extend(torch.sigmoid(model(batch_x)).cpu().numpy().flatten().tolist())
        predictions_matrix[:, model_idx] = fold_probs

    ensemble = predictions_matrix.mean(axis=1)
    rows = []
    for idx, (row, true_value, mean_probability) in enumerate(zip(kept_records, Y.numpy().flatten(), ensemble)):
        out = {
            "Experiment": experiment["id"], "Base_Encoder": experiment["base_id"], "Variant": experiment["variant_id"],
            "Index": idx, "Peptide": row["peptide"], "Origin": row.get("origin", "paper_pdf"),
            "True_Class": int(true_value), "Ensemble_Probability": float(mean_probability),
            "Special_Paper_Peptide": SPECIAL_PAPER_PEPTIDES.get(row["peptide"], ""),
        }
        for model_idx in range(len(models)):
            out[f"Fold_{model_idx}_Probability"] = float(predictions_matrix[idx, model_idx])
        rows.append(out)

    fieldnames = ["Experiment", "Base_Encoder", "Variant", "Index", "Peptide", "Origin", "True_Class", "Ensemble_Probability", "Special_Paper_Peptide"] + [f"Fold_{i}_Probability" for i in range(len(models))]
    save_rows_csv(rows, experiment["validation_csv"], fieldnames)
    print(f"Saved paper validation predictions: {experiment['validation_csv']} | rows={len(rows)}")
    return rows


In [ ]:
# -----------------------------
# Experiment construction and dimension checks
# -----------------------------
def make_experiment(base_spec, variant_spec, base_featurizer):
    experiment_id = f"{base_spec['id']}__{variant_spec['id']}"
    return {
        "id": experiment_id,
        "label": f"{base_spec['label']} {variant_spec['label']}",
        "base_id": base_spec["id"],
        "variant_id": variant_spec["id"],
        "expected_base_dim": base_spec["expected_base_dim"],
        "expected_dim": base_spec["expected_base_dim"] + variant_spec["extra_dim"],
        "model_prefix": f"{base_spec['model_prefix']}_{variant_spec['id']}",
        "cv_csv": PROJECT_DIR / f"SIHA_CV_Binding_Predictions_{experiment_id}.csv",
        "validation_csv": PROJECT_DIR / f"SIHA_Paper_Validation_Results_{experiment_id}.csv",
        "featurizer": make_contextual_featurizer(
            base_featurizer,
            add_allele=variant_spec["add_allele"],
            add_phosphosite=variant_spec["add_phosphosite"],
        ),
    }


def assert_feature_dimensions(experiments, sample_record):
    for experiment in experiments:
        observed_dim = len(experiment["featurizer"](sample_record))
        assert observed_dim == experiment["expected_dim"], (
            f"{experiment['id']} feature dim mismatch: expected {experiment['expected_dim']}, observed {observed_dim}"
        )
    print(f"Feature dimension assertions passed for {len(experiments)} experiments.")


def build_all_experiments(device):
    experiments = []
    for base_spec in AE_ENCODERS:
        centroids = load_model_and_extract_centroids(base_spec["weights"], IMAGE_DIR, device)
        base_featurizer = make_autoencoder_featurizer(centroids)
        for variant_spec in FEATURE_VARIANTS:
            experiments.append(make_experiment(base_spec, variant_spec, base_featurizer))

    discrete_featurizers = {"onehot23": one_hot_23_featurizer, "blosum62_23": blosum62_23_featurizer}
    for base_spec in DISCRETE_ENCODERS:
        for variant_spec in FEATURE_VARIANTS:
            experiments.append(make_experiment(base_spec, variant_spec, discrete_featurizers[base_spec["id"]]))

    assert_feature_dimensions(experiments, siha_cv_records[0])
    return experiments


In [ ]:
# -----------------------------
# Run full ablation
# -----------------------------
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"System hardware: {device}")

all_experiments = build_all_experiments(device)
print("Ablation experiments:")
for experiment in all_experiments:
    print(f"  {experiment['id']}: dim={experiment['expected_dim']} -> {experiment['label']}")

experiment_outputs = {}
for experiment in all_experiments:
    print("\n" + "=" * 90)
    print(f"Running {experiment['id']} | {experiment['label']}")
    print("=" * 90)
    models = run_fivefold_cv(
        siha_cv_records,
        experiment["featurizer"],
        device,
        experiment,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        lr=LEARNING_RATE,
    )
    validation_rows = predict_validation(
        paper_validation_records,
        experiment["featurizer"],
        models,
        device,
        experiment,
    )
    experiment_outputs[experiment["id"]] = {"models": models, "paper_validation": validation_rows}


In [ ]:
# -----------------------------
# Analysis helpers and summary CSV generation
# -----------------------------
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, auc, confusion_matrix, f1_score, precision_score, recall_score, roc_curve


def safe_fold_auc(y_true, y_prob):
    if len(set(y_true)) < 2:
        return np.nan
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    return auc(fpr, tpr)


def summarize_cv_predictions(csv_path, title, phospho_only=False):
    df = pd.read_csv(csv_path)
    if phospho_only:
        df = df[df["Peptide"].map(is_phospho)].copy()
    df["Predicted_Class"] = (df["Probability"] >= 0.5).astype(int)
    fold_aucs = []
    for fold in sorted(df["Fold"].unique()):
        fold_df = df[df["Fold"] == fold]
        fold_aucs.append(safe_fold_auc(fold_df["True_Class"].tolist(), fold_df["Probability"].tolist()))
    valid_aucs = [value for value in fold_aucs if not np.isnan(value)]
    return {
        "Experiment": df["Experiment"].iloc[0],
        "Base_Encoder": df["Base_Encoder"].iloc[0],
        "Variant": df["Variant"].iloc[0],
        "Model": title,
        "Rows": len(df),
        "Positives": int(df["True_Class"].sum()),
        "Negatives": int((df["True_Class"] == 0).sum()),
        "Mean_AUC": float(np.mean(valid_aucs)) if valid_aucs else np.nan,
        "Std_AUC": float(np.std(valid_aucs)) if valid_aucs else np.nan,
        "Accuracy": accuracy_score(df["True_Class"], df["Predicted_Class"]),
        "Precision": precision_score(df["True_Class"], df["Predicted_Class"], zero_division=0),
        "Recall": recall_score(df["True_Class"], df["Predicted_Class"], zero_division=0),
        "F1": f1_score(df["True_Class"], df["Predicted_Class"], zero_division=0),
    }


def plot_cv_summary(csv_path, title):
    df = pd.read_csv(csv_path)
    df["Predicted_Class"] = (df["Probability"] >= 0.5).astype(int)
    fig = plt.figure(figsize=(18, 5))
    ax1 = fig.add_subplot(1, 3, 1)
    mean_fpr = np.linspace(0, 1, 100)
    tprs, aucs = [], []
    for fold in sorted(df["Fold"].unique()):
        fold_data = df[df["Fold"] == fold]
        fpr, tpr, _ = roc_curve(fold_data["True_Class"], fold_data["Probability"])
        fold_auc = auc(fpr, tpr)
        aucs.append(fold_auc)
        interp_tpr = np.interp(mean_fpr, fpr, tpr)
        interp_tpr[0] = 0.0
        tprs.append(interp_tpr)
        ax1.plot(fpr, tpr, lw=1, alpha=0.55, label=f"Fold {fold} AUC={fold_auc:.3f}")
    mean_tpr = np.mean(tprs, axis=0)
    mean_tpr[-1] = 1.0
    mean_auc = auc(mean_fpr, mean_tpr)
    std_auc = np.std(aucs)
    ax1.plot(mean_fpr, mean_tpr, color="blue", lw=2, label=f"Mean AUC={mean_auc:.3f} +/- {std_auc:.3f}")
    ax1.plot([0, 1], [0, 1], linestyle="--", color="red", alpha=0.8)
    ax1.set_title(f"{title}: ROC")
    ax1.set_xlabel("False Positive Rate")
    ax1.set_ylabel("True Positive Rate")
    ax1.legend(loc="lower right")

    ax2 = fig.add_subplot(1, 3, 2)
    cm = confusion_matrix(df["True_Class"], df["Predicted_Class"])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax2,
                xticklabels=["Non-binder", "Binder"], yticklabels=["Non-binder", "Binder"])
    ax2.set_title(f"{title}: Confusion Matrix")
    ax2.set_xlabel("Predicted")
    ax2.set_ylabel("True")

    ax3 = fig.add_subplot(1, 3, 3)
    sns.kdeplot(data=df[df["True_Class"] == 0]["Probability"], fill=True, color="red", label="True 0", ax=ax3)
    sns.kdeplot(data=df[df["True_Class"] == 1]["Probability"], fill=True, color="blue", label="True 1", ax=ax3)
    ax3.axvline(0.5, color="black", linestyle="--")
    ax3.set_xlim(0, 1)
    ax3.set_title(f"{title}: Probability Distribution")
    ax3.legend()
    plt.tight_layout()
    plt.show()


def summarize_validation_predictions(csv_path, title):
    df = pd.read_csv(csv_path)
    df["Predicted_Class"] = (df["Ensemble_Probability"] >= 0.5).astype(int)
    detected = int((df["Ensemble_Probability"] >= 0.5).sum())
    return {
        "Experiment": df["Experiment"].iloc[0],
        "Base_Encoder": df["Base_Encoder"].iloc[0],
        "Variant": df["Variant"].iloc[0],
        "Model": title,
        "Rows": len(df),
        "Predicted_Binder_ge_0_5": detected,
        "Detection_Rate": detected / len(df),
        "Mean_Probability": df["Ensemble_Probability"].mean(),
        "Median_Probability": df["Ensemble_Probability"].median(),
    }


def plot_validation_summary(csv_path, title):
    df = pd.read_csv(csv_path)
    df["Predicted_Class"] = (df["Ensemble_Probability"] >= 0.5).astype(int)
    plt.figure(figsize=(9, 6))
    ax = plt.gca()
    sns.violinplot(y="Ensemble_Probability", data=df, color="lightgray", inner="quartile", ax=ax)
    sns.swarmplot(y="Ensemble_Probability", data=df, color="darkblue", alpha=0.8, size=5, ax=ax)

    present_special = df[df["Peptide"].isin(SPECIAL_PAPER_PEPTIDES)]
    for offset_idx, (_, special_row) in enumerate(present_special.iterrows()):
        y_value = special_row["Ensemble_Probability"]
        label = special_row["Special_Paper_Peptide"] or special_row["Peptide"]
        ax.scatter(0.08, y_value, s=120, marker="D", color="gold", edgecolor="black", zorder=5)
        ax.annotate(label, xy=(0.08, y_value), xytext=(0.28, min(1.0, y_value + 0.05 + 0.03 * offset_idx)),
                    arrowprops={"arrowstyle": "->", "lw": 1}, fontsize=9, va="center")
    absent_special = sorted(set(SPECIAL_PAPER_PEPTIDES) - set(df["Peptide"]))
    if absent_special:
        print("Special paper peptides not present in this validation file:", absent_special)
    ax.axhline(0.5, color="red", linestyle="--", linewidth=2, label="Decision threshold")
    ax.set_title(title)
    ax.set_ylabel("Ensemble prediction probability")
    ax.set_xticks([])
    ax.set_xlim(-0.55, 1.05)
    ax.set_ylim(-0.05, 1.05)
    ax.legend(loc="lower right")
    plt.tight_layout()
    plt.show()
    print("Top high-probability paper peptides")
    display(df.sort_values("Ensemble_Probability", ascending=False).head(15))
    print("Top low-probability paper peptides")
    display(df.sort_values("Ensemble_Probability", ascending=True).head(15))


def special_peptide_rows(experiment, validation_csv):
    df = pd.read_csv(validation_csv)
    rows = []
    for peptide, label in SPECIAL_PAPER_PEPTIDES.items():
        hit = df[df["Peptide"] == peptide]
        if len(hit) == 0:
            rows.append({"Experiment": experiment["id"], "Base_Encoder": experiment["base_id"], "Variant": experiment["variant_id"], "Peptide": peptide, "Special_Label": label, "Present": False, "Ensemble_Probability": np.nan, "Predicted_Class": np.nan})
        else:
            row = hit.iloc[0]
            probability = float(row["Ensemble_Probability"])
            rows.append({"Experiment": experiment["id"], "Base_Encoder": experiment["base_id"], "Variant": experiment["variant_id"], "Peptide": peptide, "Special_Label": label, "Present": True, "Ensemble_Probability": probability, "Predicted_Class": int(probability >= 0.5)})
    return rows


In [ ]:
# -----------------------------
# Run HLA_B40-style plots and write ablation summaries
# -----------------------------
cv_summary_rows = []
cv_phospho_summary_rows = []
validation_summary_rows = []
special_summary_rows = []

for experiment in all_experiments:
    title = experiment["label"]
    cv_summary_rows.append(summarize_cv_predictions(experiment["cv_csv"], title, phospho_only=False))
    cv_phospho_summary_rows.append(summarize_cv_predictions(experiment["cv_csv"], title, phospho_only=True))
    validation_summary_rows.append(summarize_validation_predictions(experiment["validation_csv"], title))
    special_summary_rows.extend(special_peptide_rows(experiment, experiment["validation_csv"]))

    if PLOT_EACH_MODEL:
        plot_cv_summary(experiment["cv_csv"], title)
        plot_validation_summary(experiment["validation_csv"], f"Paper validation: {title}")

cv_summary_df = pd.DataFrame(cv_summary_rows).sort_values("Mean_AUC", ascending=False)
cv_phospho_summary_df = pd.DataFrame(cv_phospho_summary_rows).sort_values("Mean_AUC", ascending=False)
validation_summary_df = pd.DataFrame(validation_summary_rows).sort_values("Detection_Rate", ascending=False)
special_summary_df = pd.DataFrame(special_summary_rows).sort_values(["Peptide", "Experiment"])

cv_summary_df.to_csv("SIHA_Ablation_CV_Summary.csv", index=False)
cv_phospho_summary_df.to_csv("SIHA_Ablation_CV_PhosphoOnly_Summary.csv", index=False)
validation_summary_df.to_csv("SIHA_Ablation_Paper_Validation_Summary.csv", index=False)
special_summary_df.to_csv("SIHA_Ablation_Special_Peptides.csv", index=False)

print("Saved SIHA_Ablation_CV_Summary.csv")
print("Saved SIHA_Ablation_CV_PhosphoOnly_Summary.csv")
print("Saved SIHA_Ablation_Paper_Validation_Summary.csv")
print("Saved SIHA_Ablation_Special_Peptides.csv")

print("CV summary")
display(cv_summary_df)
print("CV phospho-only summary")
display(cv_phospho_summary_df)
print("Paper validation summary")
display(validation_summary_df)
print("Special peptide summary")
display(special_summary_df)
